# Entrenamiento — MobileNetV2

**Módulo 1 — Modelo de comparación**

Entrenamiento de MobileNetV2 para clasificación de productos por imagen.
Objetivo: obtener un modelo baseline liviano para comparar contra EfficientNet-B0.

In [ ]:
# Montar Drive y configurar entorno (Colab)
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tensorflow polars

import sys
sys.path.insert(0, '/content/drive/MyDrive/smartretail360')

In [ ]:
import tensorflow as tf
import polars as pl
from pathlib import Path

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 1. Configuración

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/smartretail360/data/processed/fashion_products')
SAVE_PATH = Path('/content/drive/MyDrive/smartretail360/models/image_classifier/mobilenetv2.keras')

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5  # masterCategory
SEED = 42

## 2. Carga del dataset

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / 'train'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / 'val'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=SEED,
)
print('Clases:', train_ds.class_names)

## 3. Data augmentation

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomBrightness(0.2),
], name='augmentation')

train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y)).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 4. Fase 1 — Entrenar solo la cabeza

In [ ]:
from src.image_classifier.model_mobilenetv2 import build_model

model = build_model(num_classes=NUM_CLASSES, freeze_backbone=True)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

history_head = model.fit(train_ds, validation_data=val_ds, epochs=10)

## 5. Fase 2 — Fine-tuning completo

In [ ]:
for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history_full = model.fit(train_ds, validation_data=val_ds, epochs=10)

## 6. Evaluación

In [ ]:
# TODO: cargar test set y calcular métricas con src/evaluation/metrics.py
# from src.evaluation.metrics import compute_metrics
# from src.evaluation.confusion_matrix import plot_confusion_matrix
# from src.evaluation.viability import assess_viability

## 7. Guardar modelo

In [ ]:
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(str(SAVE_PATH))
print(f'Modelo guardado en {SAVE_PATH}')